# IBMGO · Telekom regresyon test seçimi

**Bir kampanya değiştiğinde hangi testleri tekrar çalıştıralım?**

Önce kayıtlı ana deneyin genel özetini açın. Ardından tek bir havuzu inceleyin. Yeni bir hesaplama için sonuç panelindeki **Teknik ayrıntılar · Yeni deney ayarları** bölümünü kullanın.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / 'kod').is_dir():
    raise RuntimeError('Notebook’u IBMGO proje klasöründe açın.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from IPython.display import display
from kod.deneyler.sunum_deneyi import Settings, PresentationExperiment
from kod.raporlama.sunum_sonuclari import display_result

AYARLAR = Settings(sizes=[250], data_seeds=[83], algorithm_seeds=[42])
deney = PresentationExperiment(AYARLAR, root=ROOT)
try:
    from kod.raporlama.sunum_paneli import PresentationPanel
    panel = PresentationPanel(deney)
except ImportError:
    panel = None

## 1 · Problem ve telekom örneği
**12 ay %20 indirim** kampanyasında ilk aktivasyon bir kayıt oluşturur. Aynı istek kimliği yeniden gönderilince ikinci bir kayıt oluşmamalıdır. İlk fatura indirimi de doğru uygulanmalıdır.

Fatura ve tekrar istek testlerinin ön koşulu aktivasyon testidir. Yalnız süreyi azaltmak bu ilişkileri bozabilir.

## 2 · Veri ve sunum kopyası

Her testin kapsam hedefleri, tahmini süresi ve ön koşulları var. Veri tohumu bağlam ve süreleri belirler. Sonuç panelinin **Teknik ayrıntılar · Veri kopyasını düzenle** bölümünde süre, kapsam ve kritiklik değiştirilebilir.

Veriyi değiştirdikten sonra **Veri değişikliğini uygula**, ardından **Sonuçları güncelle** düğmesine basın.

## 3 · Karar değişkeni
Her test için $x_i \in \{0,1\}$. 1 seçildi, 0 seçilmedi demektir.

Örneğin $x_{fatura}=1$ ise aktivasyon da seçilmelidir. Paket önce seçilir, sonra ön koşulları koruyan bir sıraya konur.

## 4 · Zorunlu kısıtlar
- Kritik gereksinim kapsamı %100.
- Genel gereksinim kapsamı en az %90. Panelden değiştirilebilir.
- Ön koşul ihlali sıfır. Ebeveyn test indeksi çocuk testten küçük olmalıdır.

Bütün havuzlar arama başlamadan kontrol edilir. Kapsanamayan kritik hedef, olanaksız genel kapsam veya sıralı olmayan bağımlılık grafiği hata mesajıyla bildirilir.

## 5 · Amaç fonksiyonu

**Amaç puanı — düşük daha iyi.** Süre ve test sayısının tüm havuza oranı ile kapsam tekrarı birlikte küçültülür.

$$F=\frac{w_t(T_s/T)+w_n(N_s/N)+w_r R+w_q Q}{w_t+w_n+w_r+w_q}.$$

Ana ayarlar süre %40, test sayısı %40, tekrar %20 ve nadir kayıp %0. $R$ tekrarlı kapsam isabetlerinin toplam isabete oranı, $Q$ ise kapsanmayan kritik olmayan hedeflerin nadirlik ağırlıklı kaybıdır.

## 6 · MGO → BMGO → IBMGO
**MGO** sürekli uzayda dört hareketle aday üretir. **BMGO** konum farkını V-dönüşümüyle bit çevirme olasılığına dönüştürür. Ardından kapsam ve ön koşullar onarılır, gereksiz testler budanır.

**Bu projedeki IBMGO**, çeşitli başlangıç ve yerel aramayı boyuta duyarlı hareket seçenekleriyle birleştiren probleme özgü uyarlamadır. Özgün MGO veya bir makalenin birebir uygulaması olarak sunulmaz.

Boyut ≤250 için adaptif bit dönüşümü ve tekli takas, 251–750 için referans hareketleri / ICMIC / doğrusal olmayan kontrol ve çoklu komşuluk, 751+ için %50 hareket karışımı ve tekli komşuluk kullanılır. Politika ve veri üreticisi bu düzenlemede değiştirilmedi.

## 7 · Deney planı

Ana arşiv **50, 100, 250, 500, 1000 test** ve **83, 97, 109 veri tohumlarını** kapsar. Her havuzda IBMGO ile GA 10 kez, Greedy bir kez çalıştırılmıştır. Toplam **315 koşum** vardır.

Yeni deneyin ayarları ve planlanan bütçesi sonuç panelinin kapalı teknik bölümündedir. Varsayılan yeni deney 250 test, veri tohumu 83 ve algoritma tohumu 42 kullanır.

## 8 · Sonuç ekranı

**Ana deneyin genel özeti** sekmesi bütün büyüklükleri birlikte gösterir. **Tek havuzun ayrıntısı** sekmesinde sırayla deney özeti, yöntem karşılaştırması, önce–sonra kazanımı ve iterasyon görünümü bulunur.

Açılışta kayıtlı arşiv gösterilir. Yeni hesaplama **Sonuçları güncelle** düğmesiyle başlar. Filtreler mevcut sonuçları gösterir.

In [2]:
if panel is not None:
    panel.archive()
else:
    deney.load_archive()

In [3]:
if panel is not None:
    display(panel.result_view)
else:
    display_result(deney.result)

## 9 · Sonuçları anlatırken

Süreler sentetik test sürelerinin toplamıdır. Puanları aynı test büyüklüğü içinde karşılaştırın. Kayıtlı ana deney ve yeni çalıştırma panelde ayrı kaynaklar olarak gösterilir.

### Widgetsız kullanım

Aşağıdaki kapalı kod hücresinde `CALISTIR = True` seçilirse üstteki `AYARLAR` ile yeni deney başlar. Sonuçlar aynı dört bölümle gösterilir.

In [4]:
CALISTIR = False
if CALISTIR:
    deney.settings = AYARLAR
    yeni = deney.run(lambda tamam, toplam, ad: print(f'{tamam}/{toplam} koşum tamamlandı'))
    anahtar = next(k for k, r in yeni['runs'].items() if r['method'] == 'IBMGO')
    display_result(yeni, size=AYARLAR.sizes[0], key=anahtar)